In [29]:
from rich.console import Console
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

In [30]:
load_dotenv(override=True)  # Load environment variables from .env file

True

In [31]:
def show(text, style="bold green"):
    try:
        Console().print(text, style=style)
    except Exception as e:
        print(text)

In [32]:
gemini_api_key=os.getenv("GEMINI_API_KEY1")
gemini_base_url=os.getenv("GEMINI_BASE_URL")

gemini =OpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

gemini_model="gemini-3.6-flash"


In [33]:
checklist=[]
completed=[]


In [34]:
def get_checklist_report()->str:
    result=""
    for index,item in enumerate(checklist):
        if completed[index]:
            result+=f"[green]✓[/green] {item}\n"
        else:
            result+=f"[red]✗[/red] {item}\n"
    show(result)
    return result


In [35]:
get_checklist_report()

''

In [36]:
def create_checklist(description :list[str])->str:
    checklist.extend(description)
    completed.extend([False]*len(description))
    return get_checklist_report()




In [37]:
def mark_complete( index:int, completion_note:str)->str:
    if 1<= index <= len(checklist):
        completed[index-1]=True
    else:
        show(f"[red]Invalid index {index}. Please provide a valid index between 1 and {len(checklist)}.[/red]")
    Console().print(f"[blue]Completion Note:[/blue] {completion_note}")
    return get_checklist_report()

In [38]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])


✗ Buy groceries
✗ Finish week 1
✗ Eat banana

'[red]✗[/red] Buy groceries\n[red]✗[/red] Finish week 1\n[red]✗[/red] Eat banana\n'

In [39]:
mark_complete(1, "Bought")

Completion Note: Bought

✓ Buy groceries
✗ Finish week 1
✗ Eat banana

'[green]✓[/green] Buy groceries\n[red]✗[/red] Finish week 1\n[red]✗[/red] Eat banana\n'

In [40]:
create_checklist_json={
    "name": "create_checklist",
    "description":"add new checklist from list of descriptions and return the full list",
    "parameters": {

        "type": "object",
        "properties": {
            "description": {
                "type": "array",
                "items": {
                    "type": "string"
                },
                "title": "Description of checklist items"
            }
        },
        "required": ["description"]
    }

}

In [41]:
mark_complete_json={
    "name": "mark_complete",
    "description":"mark a checklist item as complete and return the full list",
    "parameters": {
        "type":"object",
        "properties": {
            "index": {
                "type": "integer",
                "title": "Index",
                "description": "the 1-based index of the checklist item to mark as complete"
            },
            "completion_note": {
                "type": "string",
                "title": "completion_note",
                "description": "a note about the completion of the checklist item"
            }
        },
        "required": ["index", "completion_note"]

    
}
}

In [42]:
tools =[ { "type": "function", "function": create_checklist_json},
         { "type": "function", "function": mark_complete_json}
        ]

In [43]:
def handle_tool_call(tool_calls):
    results=[]
    for tool_call in tool_calls:
        tool_name= tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool= globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool", "content":json.dumps(result),"tool_call_id":tool_call.id})

    return results





In [44]:
def loop(messages):
    response = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages,
        tools=tools
    )
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls 
        results= handle_tool_call(tool_calls)
        messages.append(message)
        messages.extend(results)
        response=gemini.chat.completions.create(
            model=gemini_model,
            messages=messages,
            tools=tools
            )
    show(response.choices[0].message.content)


In [45]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

In [46]:
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [47]:
checklist,completed=[],[]
loop(messages)

✗ Identify given information and estimate the missing distance between Boston and New York
✗ Set up the relative motion/distance equation for the two trains
✗ Solve the equation for the travel time
✗ Determine the exact time of day when the trains meet
✗ Format the final solution with Rich console markup

Completion Note: Given: Train A leaves Boston at 2:00 PM at 60 mph. Train B leaves NY at 3:00 PM at 80 mph. 
Estimated distance between Boston and New York is ~215 miles (rail distance ~230 miles; using 215 miles).

✓ Identify given information and estimate the missing distance between Boston and New York
✗ Set up the relative motion/distance equation for the two trains
✗ Solve the equation for the travel time
✗ Determine the exact time of day when the trains meet
✗ Format the final solution with Rich console markup

Completion Note: Set up relative speed after 3:00 PM (140 mph combined). Remaining distance at 3:00 PM is 155 miles
(based on 215 miles total distance).

✓ Identify given information and estimate the missing distance between Boston and New York
✓ Set up the relative motion/distance equation for the two trains
✗ Solve the equation for the travel time
✗ Determine the exact time of day when the trains meet
✗ Format the final solution with Rich console markup

...

Completion Note: Calculated meeting time: approximately 4:06 PM (or 4:06:26 PM) based on 215 miles distance.

✓ Identify given information and estimate the missing distance between Boston and New York
✓ Set up the relative motion/distance equation for the two trains
✓ Solve the equation for the travel time
✓ Determine the exact time of day when the trains meet
✗ Format the final solution with Rich console markup

Completion Note: Formatted final output using Rich console markup without code blocks.

✓ Identify given information and estimate the missing distance between Boston and New York
✓ Set up the relative motion/distance equation for the two trains
✓ Solve the equation for the travel time
✓ Determine the exact time of day when the trains meet
✓ Format the final solution with Rich console markup

Problem Solution: Train Meeting Time

Step 1: Given Information & Distance Estimation
• Train A (Boston to NY): Departs at 2:00 PM at 60 mph.
• Train B (NY to Boston): Departs at 3:00 PM at 80 mph.
• Distance between Boston and New York: Not specified in the question. A standard distance estimate between Boston,
MA and New York, NY is 215 miles (driving/direct rail corridor distance is typically ~215–230 miles).

---

Step 2: Distance Covered Before 3:00 PM
• From 2:00 PM to 3:00 PM (1 hour), Train A travels alone:
  Distance = 60 mph × 1 hr = 60 miles

• Remaining distance between Train A and Train B at 3:00 PM:
  215 miles - 60 miles = 155 miles

---

Step 3: Relative Motion After 3:00 PM
• At 3:00 PM, both trains are traveling toward each other.
• Combined Closing Speed: 60 mph + 80 mph = 140 mph

• Time to cover remaining distance:
  Time (t) = 155 miles / 140 mph = 31 / 28 hours ≈ 1.1071 hours
  1.1071 hours = 1 hour + 6 minutes + 26 seconds

---

Step 4: Final Meeting Time
• Adding 1 hour, 6 minutes, and 26 seconds to 3:00 PM:
  3:00 PM + 1 hr 6 min 26 sec = 4:06:26 PM

---

Conclusion:
Assuming a distance of 215 miles between Boston and New York, the two trains will meet at approximately 4:06 PM 
(4:06:26 PM). 

(Note: If using the Amtrak Northeast Corridor distance of ~230 miles, the meeting time would be approximately 4:13 
PM.)